In [3]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [2]:
# all the dependencies
import gym
from ray.rllib.algorithms.ppo import PPOConfig
from ray.rllib.algorithms.algorithm import Algorithm
from ray.tune.logger import pretty_print
import numpy as np
import os
import pandas as pd
import matplotlib as plt

C:\Users\ritwi\anaconda3\envs\total_robotics\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-06-18 08:06:16,489	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2025-06-18 08:06:20,227	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [5]:
# name of the environment
env_name = "CartPole-v1"
env = gym.make("CartPole-v1", render_mode='human')  # import the Lunar Lander environment

In [1]:
# Configuring an Agent: This agent use Defatlt settings from RayRLlib 
# Name: agent_default

config = (
    # The RL algorithm
    PPOConfig()                       
    # The gym Env
    .environment(env_name)           
    # Numbers of rollout workers
    .rollouts(num_rollout_workers = 1)
    # Use "torch" for using pytorch or "tf" or "tf2" for Tensor Flow. "tf2" is not working for the time being.
    .framework("tf")
    # Numbers of evalution workers
    .evaluation(evaluation_num_workers=1)
)

config["entropy_coeff"] = 0.01


#Building the agent with the configurations mentioned above
agent = config.build()
check_point = "agents/0"
agent.save(check_point)

NameError: name 'PPOConfig' is not defined

In [ ]:
if os.path.exists('logs/logs') is False:
    columns = ['agent_name','itr_steps','rew_mean','rew_max','rew_min','entropy','entropy_coeff','policy_loss','vf_loss']
    df = pd.DataFrame(columns=columns)
    df.to_csv('logs/logs', index=False)
    print(df)

In [ ]:
# Recall the last trained agent
agent_name_list = os.listdir('agents')
agent_name_list_in_int = [int(x) for x in agent_name_list]
last_agent_num = max(agent_name_list_in_int)
check_point = 'agents/' + str(last_agent_num)
agent = Algorithm.from_checkpoint(check_point)

# Recall the log data frame
logs = pd.read_csv('logs/logs')

training_range = 10

for i in range(last_agent_num, (last_agent_num + training_range)):
    # Train the agent
    result = agent.train()
    # Save the agent
    check_point_path_name = 'agents/' + str(i+1)
    agent.save(check_point_path_name)
    # Capture the log    
    #print(pretty_print(result)) # It gives all the parameters regarding the training
    logs.at[i,'agent_name'] = str(i+1)
    logs.at[i,'itr_steps'] = result['num_env_steps_trained_this_iter'] * (i+1)
    logs.at[i,'rew_mean'] = result['episode_reward_mean']
    logs.at[i,'rew_max'] = result['episode_reward_max']
    logs.at[i,'rew_min'] = result['episode_reward_min']
    logs.at[i,'entropy'] = result['info']['learner']['default_policy']['learner_stats']['entropy']
    logs.at[i,'entropy_coeff'] = result['info']['learner']['default_policy']['learner_stats']['entropy_coeff']
    logs.at[i,'policy_loss'] = result['info']['learner']['default_policy']['learner_stats']['policy_loss']
    logs.at[i,'vf_loss'] = result['info']['learner']['default_policy']['learner_stats']['vf_loss']
    
    print(logs)
    logs.to_csv('logs/logs', index=False)

In [ ]:
import matplotlib.pyplot as plt
logs = pd.read_csv('logs/logs')

# Plotting Column1 and Column2 with Time as the x-axis
plt.plot(logs['itr_steps'], logs['rew_mean'], label='rew_mean')
plt.plot(logs['itr_steps'], logs['rew_max'], label='rew_max')
plt.plot(logs['itr_steps'], logs['rew_min'], label='rew_min')

# Adding labels and legend
plt.xlabel('Iteration_Steps')
plt.ylabel('Values')
plt.legend()

# Display the plot
plt.show()


In [ ]:
# recall and trained agent
check_point = "agents/agent_80"
agent = Algorithm.from_checkpoint(check_point)

In [ ]:
# use the restored agent in the environment
env = gym.make("CartPole-v1", render_mode='human')
for j in range(20):
    observation = np.array(env.reset()[0])
    done = False
    score = 0
    while not done:
        action = agent.compute_single_action(observation) # here is the RL agent from RAYRLlib
        observation,reward,done,info,info = env.step(action)
        score += reward
        if (score>1000):
            done = True
    print("score:",score)
env.close()

In [60]:
agent_timesteps_total: 44000
connector_metrics:
  ObsPreprocessorConnector_ms: 0.0
  StateBufferConnector_ms: 0.0
  ViewRequirementAgentConnector_ms: 0.33803582191467285
counters:
  num_agent_steps_sampled: 44000
  num_agent_steps_trained: 44000
  num_env_steps_sampled: 44000
  num_env_steps_trained: 44000
custom_metrics: {}
date: 2023-12-04_02-16-02
done: false
episode_len_mean: 483.625
episode_media: {}
episode_reward_max: 500.0
episode_reward_mean: 483.625
episode_reward_min: 429.0
episodes_this_iter: 8
episodes_total: 8
hostname: Ritwik
info:
  learner:
    default_policy:
      custom_metrics: {}
      diff_num_grad_updates_vs_sampler_policy: 464.5
      learner_stats:
        cur_kl_coeff: 0.07500000298023224
        cur_lr: 4.999999873689376e-05
        entropy: 0.5202701091766357
        entropy_coeff: 0.0
        kl: 0.004936424549669027
        model: {}
        policy_loss: -0.021040912717580795
        total_loss: 9.85034465789795
        vf_explained_var: -0.21864642202854156
        vf_loss: 9.871016502380371
      num_agent_steps_trained: 128.0
      num_grad_updates_lifetime: 465.5
  num_agent_steps_sampled: 44000
  num_agent_steps_trained: 44000
  num_env_steps_sampled: 44000
  num_env_steps_trained: 44000
iterations_since_restore: 1
node_ip: 127.0.0.1
num_agent_steps_sampled: 44000
num_agent_steps_trained: 44000
num_env_steps_sampled: 44000
num_env_steps_sampled_this_iter: 4000
num_env_steps_sampled_throughput_per_sec: 201.4758138526751
num_env_steps_trained: 44000
num_env_steps_trained_this_iter: 4000
num_env_steps_trained_throughput_per_sec: 201.4758138526751
num_faulty_episodes: 0
num_healthy_workers: 1
num_in_flight_async_reqs: 0
num_remote_worker_restarts: 0
num_steps_trained_this_iter: 4000
perf:
  cpu_util_percent: 28.41333333333333
  ram_util_percent: 79.24666666666667
pid: 19472
policy_reward_max: {}
policy_reward_mean: {}
policy_reward_min: {}
sampler_perf:
  mean_action_processing_ms: 0.39178268577539455
  mean_env_render_ms: 0.0
  mean_env_wait_ms: 0.16466345497442167
  mean_inference_ms: 2.061613676876344
  mean_raw_obs_processing_ms: 0.8068118682952858
sampler_results:
  connector_metrics:
    ObsPreprocessorConnector_ms: 0.0
    StateBufferConnector_ms: 0.0
    ViewRequirementAgentConnector_ms: 0.33803582191467285
  custom_metrics: {}
  episode_len_mean: 483.625
  episode_media: {}
  episode_reward_max: 500.0
  episode_reward_mean: 483.625
  episode_reward_min: 429.0
  episodes_this_iter: 8
  hist_stats:
    episode_lengths: [500, 500, 500, 494, 500, 500, 429, 446]
    episode_reward: [500.0, 500.0, 500.0, 494.0, 500.0, 500.0, 429.0, 446.0]
  num_faulty_episodes: 0
  policy_reward_max: {}
  policy_reward_mean: {}
  policy_reward_min: {}
  sampler_perf:
    mean_action_processing_ms: 0.39178268577539455
    mean_env_render_ms: 0.0
    mean_env_wait_ms: 0.16466345497442167
    mean_inference_ms: 2.061613676876344
    mean_raw_obs_processing_ms: 0.8068118682952858
time_since_restore: 19.866973876953125
time_this_iter_s: 19.866973876953125
time_total_s: 19.866973876953125
timers:
  learn_throughput: 667.626
  learn_time_ms: 5991.381
  load_throughput: 0.0
  load_time_ms: 0.0
  sample_time_ms: 13853.882
  synch_weights_time_ms: 8.237
  training_iteration_time_ms: 19853.5
timestamp: 1701636362
timesteps_total: 44000
training_iteration: 11
trial_id: default

SyntaxError: leading zeros in decimal integer literals are not permitted; use an 0o prefix for octal integers (3522548778.py, line 12)

In [75]:
print(result['info']['learner']['default_policy']['learner_stats']['entropy'])
print(result['info']['learner']['default_policy']['learner_stats']['entropy_coeff'])
print(result['info']['learner']['default_policy']['learner_stats']['policy_loss'])
print(result['info']['learner']['default_policy']['learner_stats']['vf_loss'])
print(result['info']['learner']['default_policy']['learner_stats']['cur_lr'])

0.5069775
0.0
-0.020565892
9.790593


In [64]:
result

{'custom_metrics': {},
 'episode_media': {},
 'info': {'learner': {'default_policy': {'learner_stats': {'cur_kl_coeff': 0.00937500037252903,
     'cur_lr': 4.999999873689376e-05,
     'total_loss': 9.770115,
     'policy_loss': -0.020565892,
     'vf_loss': 9.790593,
     'vf_explained_var': -0.14465216,
     'kl': 0.0092059625,
     'entropy': 0.5069775,
     'entropy_coeff': 0.0,
     'model': {}},
    'custom_metrics': {},
    'num_agent_steps_trained': 128.0,
    'num_grad_updates_lifetime': 465.5,
    'diff_num_grad_updates_vs_sampler_policy': 464.5}},
  'num_env_steps_sampled': 84000,
  'num_env_steps_trained': 84000,
  'num_agent_steps_sampled': 84000,
  'num_agent_steps_trained': 84000},
 'sampler_results': {'episode_reward_max': 500.0,
  'episode_reward_min': 354.0,
  'episode_reward_mean': 472.5,
  'episode_len_mean': 472.5,
  'episode_media': {},
  'episodes_this_iter': 8,
  'policy_reward_min': {},
  'policy_reward_max': {},
  'policy_reward_mean': {},
  'custom_metrics': {